# HAR Real-Data Experiment

This notebook reproduces the HAR real-data experiment used in the paper.

Protocol summary:
- raw 561-dimensional HAR features with global Min-Max scaling
- 30 subject-specific tasks
- binary label: `standing` versus all other activities
- 20% held-out test split per task
- 30 random train/test splits
- 5-fold cross-validation for ARMUL and OURS

The implementation is delegated to `real_data_har.py` so that the notebook and the script stay synchronized.


In [ ]:
from pathlib import Path
import sys


def _running_in_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
    except Exception:
        return False
    return True


if _running_in_colab():
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

_code_candidates = [
    Path.cwd(),
    Path.cwd() / "MTLR_Codes",
    Path("/content/drive/MyDrive/Colab Notebooks/MTLR_Codes"),
    Path("/content/drive/My Drive/Colab Notebooks/MTLR_Codes"),
    Path("/content/drive/MyDrive/MTLR_Codes"),
    Path("/content/drive/My Drive/MTLR_Codes"),
]

CODE_DIR = None
for candidate in _code_candidates:
    if (candidate / "path_setup.py").exists():
        CODE_DIR = candidate.resolve()
        break

if CODE_DIR is None:
    raise FileNotFoundError(
        "Could not locate MTLR_Codes. In Colab, upload the folder to "
        "MyDrive/Colab Notebooks/MTLR_Codes."
    )

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from path_setup import setup_project_paths

CODE_DIR, PROJECT_ROOT, FIGURE_DIR = setup_project_paths(chdir=True)
print(f"CODE_DIR    : {CODE_DIR}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"FIGURE_DIR  : {FIGURE_DIR}")

import importlib
import pandas as pd

pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 20)

import real_data_har as har
importlib.reload(har)

print(f"Positive HAR label(s): {har.POSITIVE_LABELS}")
print(f"Default q-grid: {har.Q_GRID}")


Mounted at /content/drive
CODE_DIR    : /content/drive/MyDrive/Colab Notebooks/MTLR_Codes
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks
FIGURE_DIR  : /content/drive/MyDrive/Colab Notebooks/MTLR_Codes/Images
Positive HAR label(s): [5]
Default q-grid: [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]


In [ ]:
DATA_DIR = CODE_DIR
Q_GRID = list(har.Q_GRID)
N_SPLITS = har.N_SPLITS
N_FOLD = har.N_FOLD
ETA = har.ETA
MAXITER = har.MAXITER

# For a quick Colab smoke test, temporarily set `N_SPLITS = 1`.
har.Q_GRID = Q_GRID
har.N_SPLITS = N_SPLITS
har.N_FOLD = N_FOLD
har.ETA = ETA
har.MAXITER = MAXITER

print("Configuration")
print("-------------")
print(f"DATA_DIR : {DATA_DIR}")
print(f"Q_GRID   : {Q_GRID}")
print(f"N_SPLITS : {N_SPLITS}")
print(f"N_FOLD   : {N_FOLD}")
print(f"ETA      : {ETA}")
print(f"MAXITER  : {MAXITER}")


Configuration
-------------
DATA_DIR : /content/drive/MyDrive/Colab Notebooks/MTLR_Codes
Q_GRID   : [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]
N_SPLITS : 30
N_FOLD   : 5
ETA      : 0.1
MAXITER  : 200


In [ ]:
results_df = har.run_experiment(DATA_DIR)

summary = results_df[["DP", "ITL", "ARMUL", "OURS"]].agg(["mean", "std"]).T
summary.columns = ["Mean Error", "Std Dev"]
summary


Loaded HAR data with m=30 tasks and d=561 raw features.
Starting experiment: 30 random splits with 5-fold CV.
--- Split 1/30 ---
   [Split 0] ARMUL: 0.0630 | OURS: 0.0132
   [Split 1] ARMUL: 0.0484 | OURS: 0.0106
   [Split 2] ARMUL: 0.0440 | OURS: 0.0121
   [Split 3] ARMUL: 0.0498 | OURS: 0.0109
   [Split 4] ARMUL: 0.0513 | OURS: 0.0067
--- Split 6/30 ---
   [Split 5] ARMUL: 0.0532 | OURS: 0.0145
   [Split 6] ARMUL: 0.0513 | OURS: 0.0126
   [Split 7] ARMUL: 0.0537 | OURS: 0.0093
   [Split 8] ARMUL: 0.0503 | OURS: 0.0101
   [Split 9] ARMUL: 0.0523 | OURS: 0.0141
--- Split 11/30 ---
   [Split 10] ARMUL: 0.0523 | OURS: 0.0117
   [Split 11] ARMUL: 0.0562 | OURS: 0.0141
   [Split 12] ARMUL: 0.0518 | OURS: 0.0139
   [Split 13] ARMUL: 0.0552 | OURS: 0.0144
   [Split 14] ARMUL: 0.0581 | OURS: 0.0140
--- Split 16/30 ---
   [Split 15] ARMUL: 0.0498 | OURS: 0.0162
   [Split 16] ARMUL: 0.0498 | OURS: 0.0115
   [Split 17] ARMUL: 0.0449 | OURS: 0.0081
   [Split 18] ARMUL: 0.0572 | OURS: 0.0129
   [S

,Mean Error,Std Dev
DP,0.0761,0.0046
ITL,0.0467,0.0051
ARMUL,0.0524,0.0043
OURS,0.0125,0.0032
